In [ ]:
import sys; sys.path.append('..')
import MeshFEM
import sheet_convergence, sim_utils, loads, py_newton_optimizer
import elastic_solid, energy, sim_utils, mesh

import numpy as np
from matplotlib import pyplot as plt

# Plane Stress Manufactured Solutions
Analyze convergence to an analytical ground-truth Linear Elasticity solution constructed using the method of manufactured solutions.
We pick a solution field `u_gt` and `manufactured_solutions.PlaneStress` computes the corresponding body force symbolically.
Then we simulate with that body force and measure convergence to `u_gt`.

TODO: test traction boundarys (currently we apply the ground-truth boundary displacements as a Dirichlet condition).

In [ ]:
import manufactured_solutions
YOUNG = 1
POISSON = 0.3
ps_solution = manufactured_solutions.PlaneStress(E=YOUNG, nu=POISSON)

## Remeshing version

In [ ]:
def meshForResolution(maxArea, deg=1):
    return sheet_convergence.getMesh(maxArea, 1, degree=deg, embeddingDimension=2)

def equilibriumForMesh(m):
    psi = energy.IsotropicLinearElastic(2, YOUNG, POISSON)
    es = elastic_solid.ElasticSolid(m, psi)

    es.setIdentityDeformation()
    x_rest = es.getDeformedPositions()
    x = es.getVars()
    bdry_dofs = np.ravel([[2 * i, 2 * i + 1] for i in m.boundaryNodes()])
    x[bdry_dofs] += ps_solution.u(m.nodes()[m.boundaryNodes()]).ravel()
    es.setVars(x)
    
    bf = loads.BodyForce(es)
    bf.nodalForceDensity = ps_solution.f(m.nodes())

    opts = py_newton_optimizer.NewtonOptimizerOptions()
    opts.verbose = 0
    es.computeEquilibrium([bf], fixedVars=bdry_dofs, opts=opts)
    return es
    
def equilibriumForResolution(maxArea = 1, deg=1):
    m = meshForResolution(maxArea, deg=deg)
    return equilibriumForMesh(m)

In [ ]:
maxAreas = np.logspace(-2, -5, 20)
data_d1 = [equilibriumForResolution(ma) for ma in maxAreas]
data_d2 = [equilibriumForResolution(ma, deg=2) for ma in maxAreas]

In [ ]:
import field_sampler
def subsampledDeformation(es, P): return field_sampler.FieldSampler(es.mesh()).sample(P, es.getDeformedPositions())
def euclideanSolnRelError(es, P, u_gt): return np.linalg.norm((subsampledDeformation(es, P) - P) - u_gt) / np.linalg.norm(u_gt)

In [ ]:
# Pick sample points for validating the solutions
X_sample = data_d2[0].getRestPositions()
u_gt = ps_solution.u(X_sample)
errorForSolution = lambda es: euclideanSolnRelError(es, X_sample, u_gt)

In [ ]:
medianEdgeLens = [np.median(es.mesh().edgeLengths()) for es in data_d1]

In [ ]:
plt.loglog(medianEdgeLens, [errorForSolution(es) for es in data_d1], label='degree 1')
plt.loglog(medianEdgeLens, [errorForSolution(es) for es in data_d2], label='degree 2')
plt.legend()
plt.xlabel('Median h')
plt.ylabel('Displacement Error')
plt.grid()

## Subdivsion-based version

In [ ]:
import igl
def equilibriumForSubdiv(nsubdiv = 0, deg=1):
    m = sheet_convergence.getMesh(1, 1, embeddingDimension=2)
    m = mesh.Mesh(*igl.upsample(m.vertices(), m.elements(), number_of_subdivs=nsubdiv), degree=deg, embeddingDimension=2)
    return equilibriumForMesh(m)

nsubdivs=np.arange(9)
es_sub_d1 = [equilibriumForSubdiv(nsubdiv, deg=1) for nsubdiv in nsubdivs]
es_sub_d2 = [equilibriumForSubdiv(nsubdiv, deg=2) for nsubdiv in nsubdivs]

In [ ]:
plt.loglog(np.power(2.0, -nsubdivs), [errorForSolution(es) for es in es_sub_d1], label='degree 1')
plt.loglog(np.power(2.0, -nsubdivs), [errorForSolution(es) for es in es_sub_d2], label='degree 2')
plt.xlabel('h')
plt.ylabel('Displacement Error')
plt.grid()
# plt.gca().set_aspect('equal')

# Convergence to high-resolution ground truth simulation
In this experiment, we stretch a square solid whose left edge is glued in place.
Unfortunately, the solution has stress singularities at the top- and bottom-left corners that degrade convergence.

In [ ]:
STRETCH_MAGNITUDE = 0.05

In [ ]:
def meshForResolution(maxArea, deg=1):
    return sheet_convergence.getMesh(maxArea, 1, degree=deg, embeddingDimension=2)
    
def equilibriumForResolution(maxArea = 1, deg=1, psi = energy.IsotropicLinearElastic(2, 1, -0.3)):
    m = meshForResolution(maxArea, deg=deg)
    es = elastic_solid.ElasticSolid(m, psi)

    rightBCVars = sim_utils.getBBoxVars(es, sim_utils.BBoxFace.MAX_X, displacementComponents=[0])
    leftBCVars  = sim_utils.getBBoxVars(es, sim_utils.BBoxFace.MIN_X)

    es.setIdentityDeformation()
    x = es.getVars()
    x[rightBCVars] *= 1.0 + STRETCH_MAGNITUDE
    es.setVars(x)

    opts = py_newton_optimizer.NewtonOptimizerOptions()
    opts.verbose = 0
    es.computeEquilibrium(fixedVars=rightBCVars + leftBCVars, opts=opts)
    return es

In [ ]:
maxAreas = np.logspace(-2, -5.75, 20)
es_d1 = [equilibriumForResolution(ma) for ma in maxAreas]
es_d2 = [equilibriumForResolution(ma, deg=2) for ma in maxAreas]

In [ ]:
medianEdgeLens = [np.median(es.mesh().edgeLengths()) for es in es_d1]

In [ ]:
es_gt = es_d2[-1]
m_gt = es_gt.mesh()
x_gt = es_gt.getDeformedPositions()

In [ ]:
energy_gt = es_gt.energy()
def energy_error(es): return np.abs(es.energy() - energy_gt) / energy_gt

In [ ]:
plt.loglog(medianEdgeLens[:-1], [energy_error(es) for es in es_d1[:-1]], label='degree 1')
plt.loglog(medianEdgeLens[:-1], [energy_error(es) for es in es_d2[:-1]], label='degree 2')
plt.xlabel('Median h')
plt.ylabel('Energy Error')
plt.legend()
plt.grid()

In [ ]:
import field_sampler
def subsampledSolution(es): return field_sampler.FieldSampler(es.mesh()).sample(m_gt.nodes(), es.getDeformedPositions())
def euclideanSolnRelError(es): return np.linalg.norm(subsampledSolution(es) - es_gt.getDeformedPositions()) / np.linalg.norm(x_gt)

In [ ]:
plt.loglog(medianEdgeLens[:-1], [euclideanSolnRelError(es) for es in es_d1[:-1]], label='degree 1')
plt.loglog(medianEdgeLens[:-1], [euclideanSolnRelError(es) for es in es_d2[:-1]], label='degree 2')
plt.legend()
plt.xlabel('Median h')
plt.ylabel('Deformation Error')
plt.grid()

## Subdivsion version

In [ ]:
import igl

In [ ]:
def equilibriumForSubdiv(nsubdiv = 0, deg=1, psi = energy.IsotropicLinearElastic(2, 1, 0.3)):
    m = sheet_convergence.getMesh(1, 1, embeddingDimension=2)
    V, F = m.vertices(), m.elements()
    
    #for i in range(nsubdiv):
    V, F = igl.upsample(V, F, number_of_subdivs=nsubdiv)
    m = mesh.Mesh(V, F, degree=deg, embeddingDimension=2)
        
    es = elastic_solid.ElasticSolid(m, psi)

    rightBCVars = sim_utils.getBBoxVars(es, sim_utils.BBoxFace.MAX_X, displacementComponents=[0])
    leftBCVars  = sim_utils.getBBoxVars(es, sim_utils.BBoxFace.MIN_X)

    es.setIdentityDeformation()
    x = es.getVars()
    x[rightBCVars] *= 1.0 + STRETCH_MAGNITUDE
    es.setVars(x)

    opts = py_newton_optimizer.NewtonOptimizerOptions()
    opts.verbose = 0
    es.computeEquilibrium(fixedVars=rightBCVars + leftBCVars, opts=opts)
    return es

In [ ]:
es = equilibriumForSubdiv()

In [ ]:
def visualizeForSubdiv(nsubdiv = 0, deg=1, psi = energy.IsotropicLinearElastic(2, 1, -0.3)):
    es = equilibriumForSubdiv(nsubdiv, deg=deg, psi=psi)
    stresses = [np.abs(np.linalg.eigh(es.cauchyStress(i))[0]).max() for i in range(es.numElements())]
    return Viewer(es, scalarField=stresses, wireframe=False).antialiasedImage()

In [ ]:
for i in range(9):
    visualizeForSubdiv(i, deg=1).save(f'd1_stress_{i}.png')

In [ ]:
!open .

In [ ]:
nsubdivs = np.arange(10)

In [ ]:
es_sub_d1 = [equilibriumForSubdiv(nsubdiv) for nsubdiv in nsubdivs]
es_sub_d2 = [equilibriumForSubdiv(nsubdiv, deg=2) for nsubdiv in nsubdivs]

In [ ]:
es_sub_gt = es_sub_d2[-1].energy()
def energy_error_sub(es): return np.abs(es.energy() - es_sub_gt) / es_sub_gt

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
plt.loglog(np.power(2.0, -nsubdivs[:-1]), [energy_error_sub(es) for es in es_sub_d1[:-1]])
plt.loglog(np.power(2.0, -nsubdivs[:-1]), [energy_error_sub(es) for es in es_sub_d2[:-1]])
plt.grid()
#plt.axes('square')